# 01 — Trening modelu w PyTorch Lightning bez AMP

Notebook zastępuje ręczne funkcje z `engine.py` treningiem opartym o Lightning. Trening działa w FP32, bez AMP.

In [ ]:
from pathlib import Path
import sys
import shutil
import os
import json
import torch
import time
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

## Konfiguracja eksperymentu

In [ ]:
# Zmień, jeśli projekt znajduje się w innym miejscu
PROJ_DIR = Path("/home/jakjas3894/thesis")

if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))
    
from src.data_module import ImageFolderDataModule
from src.model_loader import load_model_factory
from src.losses import make_loss
from src.lit_classifier import LitImageClassifier
from src.reporting import save_training_artifacts
from src.archive import prepare_dataset_from_tar, make_training_archive
from src.callbacks import EpochMetricsLogger
from src.benchmark import collect_model_statistics, save_model_statistics

In [ ]:
DATASET_FILE_NAME = "F4K_trajectory_V1"
MODEL_NAME = "modernstem_eca_resnet50"

PD_PATH = Path(os.environ.get("PDDIRS", "/lustre/pd03/hpc-jakjas3894-1777231388"))
DATASET_TAR = PD_PATH / "datasets" / f"{DATASET_FILE_NAME}.tar"

TMP_DIR = Path(os.environ.get("TMPDIR", "/tmp"))
DATA_DIR = TMP_DIR / DATASET_FILE_NAME
MODEL_FILE = PROJ_DIR / "models" / f"{MODEL_NAME}.py"
MODEL_FACTORY_NAME = "build_model"

# archive
ARCHIVES_DIR = PD_PATH / "models" / "seeds"

In [ ]:
SEED = 777
L.seed_everything(SEED, workers=True)

IMAGE_SIZE = 288
BATCH_SIZE = 32
NUM_WORKERS = 4
EPOCHS = 50

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
OPTIMIZER_NAME = "adamw"
SCHEDULER_NAME = "cosine"

LOSS_NAME = "ce"  # ce, weighted_ce, focal, cb_focal
FOCAL_GAMMA = 2.0
CB_BETA = 0.9999

ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1

if LOSS_NAME == "focal":
    SESSION_NAME = (
        f"{MODEL_NAME}_{LOSS_NAME}_g{FOCAL_GAMMA}_"
        f"{EPOCHS}_{IMAGE_SIZE}_seed_{SEED}"
    )
elif LOSS_NAME == "cb_focal":
    SESSION_NAME = (
        f"{MODEL_NAME}_{LOSS_NAME}_g{FOCAL_GAMMA}_b{CB_BETA}_"
        f"{EPOCHS}_{IMAGE_SIZE}_seed_{SEED}"
    )
else:
    SESSION_NAME = (
        f"{MODEL_NAME}_{LOSS_NAME}_"
        f"{EPOCHS}_{IMAGE_SIZE}_seed_{SEED}"
    )

ARCHIVE_PATH = ARCHIVES_DIR / f"{SESSION_NAME}.tar.gz"


# model output
OUTPUT_DIR = TMP_DIR / f"{SESSION_NAME}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
METRICS_DIR = OUTPUT_DIR / "metrics"
LOG_DIR = OUTPUT_DIR / "logs"

# clear output
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    print(f"Czyszczenie folderu {OUTPUT_DIR}")

# create directories (create parents if needed, dont throw if already exists)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": SEED,
    "data_dir": str(DATA_DIR),
    "model_file": str(MODEL_FILE),
    "model_factory_name": MODEL_FACTORY_NAME,
    "session_name": SESSION_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "optimizer_name": OPTIMIZER_NAME,
    "scheduler_name": SCHEDULER_NAME,
    "loss_name": LOSS_NAME,
    "focal_gamma": FOCAL_GAMMA,
    "cb_beta": CB_BETA,
    "precision": "32-true",
}

# save config
with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

print("Accelerator:", ACCELERATOR)
print("Session name: ", SESSION_NAME)
print("Output dir:", OUTPUT_DIR)

In [ ]:
prepare_dataset_from_tar(DATASET_TAR, DATA_DIR)

## Dane

In [ ]:
datamodule = ImageFolderDataModule(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

datamodule.setup()

num_classes = datamodule.num_classes
class_names = datamodule.class_names
samples_per_class = datamodule.samples_per_class
class_weights = datamodule.class_weights()

print("Liczba klas:", num_classes)
print("Klasy:", class_names)
print("Liczności klas w train:", samples_per_class)

## Model i funkcja straty

In [ ]:
build_model = load_model_factory(MODEL_FILE, MODEL_FACTORY_NAME)
model = build_model(num_classes=num_classes)

criterion = make_loss(
    loss_name=LOSS_NAME,
    class_weights=class_weights,
    samples_per_class=samples_per_class,
    focal_gamma=FOCAL_GAMMA,
    cb_beta=CB_BETA,
)

lit_model = LitImageClassifier(
    model=model,
    criterion=criterion,
    num_classes=num_classes,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    optimizer_name=OPTIMIZER_NAME,
    scheduler_name=SCHEDULER_NAME,
    max_epochs=EPOCHS,
    class_names=class_names,
    config=CONFIG,
)

## Callbacki i logger

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="best-{epoch:03d}-{val_macro_f1:.4f}",
    monitor="val_macro_f1",
    mode="max",
    save_top_k=1,
    save_last=True,
)
epoch_logger = EpochMetricsLogger(
    save_path=METRICS_DIR / "epoch_metrics.csv"
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")
logger = CSVLogger(save_dir=LOG_DIR, name="", version="")

## Trening bez AMP

In [ ]:
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator=ACCELERATOR,
    devices=DEVICES,
    precision="32-true",
    logger=logger,
    callbacks=[checkpoint_callback, lr_monitor],
    log_every_n_steps=10,
    deterministic=False
)

training_start_time = time.perf_counter()
print("Start treningu:", training_start_time)

trainer.fit(lit_model, datamodule=datamodule)

training_end_time = time.perf_counter()
training_time_s = training_end_time - training_start_time
training_time_min = training_time_s / 60

print("Koniec treningu:", training_end_time)
print("Czas treningu w sekundach:", training_time_s)
print("Czas treningu w minutach:", training_time_min)

print("Najlepszy checkpoint:", checkpoint_callback.best_model_path)
print("Najlepszy val_macro_f1:", checkpoint_callback.best_model_score)

## Test najlepszego checkpointu

In [ ]:
test_results = trainer.test(
    model=lit_model,
    datamodule=datamodule,
    ckpt_path=checkpoint_callback.best_model_path,
)

with open(METRICS_DIR / "test_results.json", "w", encoding="utf-8") as f:
    json.dump(test_results, f, ensure_ascii=False, indent=2)

test_results

In [ ]:
training_time_info = {
    "training_time_s": training_time_s,
    "training_time_min": training_time_min,
    "epochs": EPOCHS,
    "avg_epoch_time_s": training_time_s / EPOCHS,
    "avg_epoch_time_min": training_time_min / EPOCHS,
}

import json

with open(METRICS_DIR / "training_time.json", "w", encoding="utf-8") as f:
    json.dump(training_time_info, f, ensure_ascii=False, indent=2)

In [ ]:
stats = collect_model_statistics(
    model=model,
    experiment_name=SESSION_NAME,
    image_size=256,
)

save_model_statistics(
    stats,
    METRICS_DIR / "model_benchmarks.csv"
)

## Zapis historii metryk

In [ ]:
history_csv = METRICS_DIR / "history.csv"
df_history = save_training_artifacts(
    csv_log_dir=logger.log_dir,
    metrics_dir=METRICS_DIR,
    test_results=test_results,
    y_true=lit_model.test_targets,
    y_pred=lit_model.test_preds,
    class_names=class_names,
    monitor_metric="val_macro_f1",
)
df_history.tail()

In [ ]:
make_training_archive(OUTPUT_DIR, ARCHIVE_PATH)